In [1]:
import pandas as pd
import numpy as np
import os
import glob as glob
import sys
import json

parent_dir = os.path.abspath(os.path.join(os.path.dirname(os.getcwd())))
sys.path.append(parent_dir)
import cpg_harmonizer
import s3_loader
import harmonize_checker

In [2]:
%load_ext autoreload
%autoreload 2

# Enter Identifiers, Per-Dataset Metadata

In [3]:
in_staging = False # True or False
profile = None # enter AWS profile if in_staging is True, otherwise None

In [4]:
project_name = "cpg0039-garcia-fossa-livecellpainting"
source_list = ['unicamp']
output_parent_directory = "/Users/eweisbar/Desktop/cpgtest"

# typically, per-dataset metadata
# if not dataset-wide, delete from here and create conditional entry below
per_dataset_manual = {
    'Plate_Size':96,
    'CP_Version':'live cell painting',
    'DOI_to_Cite':'10.1091/mbc.E24-07-0308',
    'Cell_Line_Name':'Huh7',
    'Cell_Line_Type':'hepatocyte cancer',
    'Cell_Line_Modification':'None',
    'Cell_Line_Organism':'Homo sapiens',
    'Microscope_Name':'Agilent BioTek Cytation 5',
    'Microscope_Binning': 1,
    'Microscope_Modality':'Widefield',
    'Microscope_Objective_Magnification':20,
    'Microscope_Objective_NA':.45,
    'Microscope_Pixel_Size': .321895,
    'Image_Bit_Depth': 16,
    'Image_Size_X':1224,
    'Image_Size_Y':904,
    #'Timepoint_Secondary_Treatment':0,
    'Timepoint_Acquisition':0,
    'Treatment_Category':'Compound',
}

In [18]:
ex_em_fluor_dict = {
       'Lysotracker':{'ex_peak':628, # lysotracker, only in one batch
           'ex_width':np.nan,
           'em_peak':685,
           'em_width':np.nan,
           'fluorophore':'Deep Red'},
       'AO_Green':{'ex_peak':469,
           'ex_width':np.nan,
           'em_peak':525,
           'em_width':np.nan,
           'fluorophore':'Acridine Orange'},
       'AO_Red':{'ex_peak':531,
          'ex_width':np.nan,
           'em_peak':647,
           'em_width':np.nan,
           'fluorophore':'Acridine Orange'},
       'DNA':{'ex_peak':377, # used for DAPI, only in one batch
          'ex_width':np.nan,
           'em_peak':447,
           'em_width':np.nan,
           'fluorophore':'33342'}
}

# Join CPG Metadata

In [6]:
if len(source_list) == 1:
    output_directory = os.path.join(output_parent_directory, project_name, source_list[0], "workspace", "metadata_harmonized")
else:
    output_directory = os.path.join(output_parent_directory, project_name, "all", "workspace", "metadata_harmonized")
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=True)

metadata_paths = []
load_paths = []
for src in source_list:
    metadata_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/metadata/platemaps/", in_staging=in_staging, profile=profile))
    load_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/load_data_csv/", in_staging=in_staging, profile=profile))

project = cpg_harmonizer.Project(output_directory, "../output_structure.json", project_name=project_name)

In [7]:
load_data_csvs = []
list_batch_names_from_load_data = []
for path in load_paths:
    if path.endswith("load_data.csv"):
        load_data_csvs.append(s3_loader.read_s3_file(path, sep = ",", in_staging=in_staging, profile=profile))
        list_batch_names_from_load_data.append(path.split("/")[-3])
if len(load_data_csvs) == 0:
    print("No load_data.csv files found. Please check the load_data_csv folder.")

In [8]:
platemaps = []
barcode_platemap_csvs = []
list_batch_names_from_barcode_platemaps = []
list_platemap_names = []
list_batch_names_from_platemaps = []
external_tsv = []

for path in metadata_paths:
    if path.endswith("barcode_platemap.csv"):
        barcode_platemap_csvs.append(s3_loader.read_s3_file(path, sep = ",",in_staging=in_staging, profile=profile))
        list_batch_names_from_barcode_platemaps.append(path.split("/")[-2])
    if '/platemap/' in path and path.endswith(".txt"):
        platemaps.append(s3_loader.read_s3_file(path, sep = "\t", in_staging=in_staging, profile=profile))
        list_platemap_names.append(path.split("/")[-1].split(".")[0])
        list_batch_names_from_platemaps.append(path.split("/")[-3])
    if 'external' in path:
        if '.csv' in path:
            external_tsv.append(s3_loader.read_s3_file(path, sep = ",", in_staging=in_staging, profile=profile))
        if '.tsv' in path:
            external_tsv.append(s3_loader.read_s3_file(path, sep = "\t", in_staging=in_staging, profile=profile))
if barcode_platemap_csvs == []:
    print("No barcode_platemap.csv files found. Please check the metadata/platemaps folder.")
if platemaps == []:
    print("No platemap files found. Please check the metadata/platemaps folder.")

In [9]:
all_load_data_platenames = []
for df in load_data_csvs:
    all_load_data_platenames.extend(df['Metadata_Plate'].unique())
all_barcode_platemap_platenames = []
for df in barcode_platemap_csvs:
    all_barcode_platemap_platenames.extend(df['Assay_Plate_Barcode'].unique())
if set(all_load_data_platenames) != set(all_barcode_platemap_platenames):
    print("Plate names in load_data.csv and barcode_platemap.csv do not match.")
    print('Do NOT proceed until they are matched')
    print(f"Plate names from load_data.csv: {list(set(all_load_data_platenames))}")
    print(f"Plate names from barcode_platemap.csv: {list(set(all_barcode_platemap_platenames))}")

In [10]:
if not list(set(list_batch_names_from_platemaps)) == list(set(list_batch_names_from_load_data)):
    print("Warning: Batch names in platemaps and load_data.csv do not match.")
    print(f"Batch names from platemaps: {list(set(list_batch_names_from_platemaps))}")
    print(f"Batch names from load_data.csv: {list(set(list_batch_names_from_load_data))}")
if not list(set(list_batch_names_from_platemaps)) == list(set(list_batch_names_from_barcode_platemaps)):
    print("Warning:Batch names in platemaps and barcode_platemap.csv do not match.")
    print(f"Batch names from platemaps: {list(set(list_batch_names_from_platemaps))}")
    print(f"Batch names from barcode_platemap.csv: {list(set(list_batch_names_from_barcode_platemaps))}")

In [11]:
complete_df = project.run_conversion(
    load_data_csvs = load_data_csvs, 
    load_data_csv_batch = list_batch_names_from_load_data, 
    platemap_csvs = barcode_platemap_csvs,  
    platemap_csv_batch = list_batch_names_from_barcode_platemaps, 
    platemap_txt= platemaps, 
    platemap_txt_batch = list_batch_names_from_platemaps,
    platemap_txt_name = list_platemap_names,
    external_tsv = external_tsv,
    external_merge_regex = [["compound"],["compound"]]
)

No external metadata found. Not all projects have external metadata.
Merging in barcode platemap csvs
Merging in platemaps
Skipping external metadata. Not all projects have external metadata.
No concentration columns found — skipping concentration merge.
Harmonizing values of Treatment_Control_Class
Harmonizing values of Label


# Additional cleaning steps

In [12]:
# add per-experiment metadata manually annotated above
for col, val in per_dataset_manual.items():
    complete_df[col] = val

# add source information inferred from file path
for source in source_list:
    complete_df.loc[complete_df['File Path'].str.contains(f"/{source}/"),'Source'] = source

In [13]:
with open('../inferable_relationships.json', "r") as f:
    inferable_metadata = json.load(f)

# infer label metadata from known relationships
if per_dataset_manual['CP_Version'] == 'other':
    print("Did not infer label metadata because CP_Version not inferrable")
    print(f"Labels that need to be manually declared are {complete_df['Label'].unique()}")
else:
    found_mismatch = False
    if not all([x in inferable_metadata["Label"].keys() for x in complete_df['Label'].unique()]):
        for x in complete_df['Label'].unique():
            if x not in inferable_metadata["Label"].keys():
                for key, value in inferable_metadata["Label_Alternative_Names"].items():
                    if x in value:
                        print(f"Inferred label {x} to be {key} based on alternative names")
                        complete_df.loc[complete_df['Label'] == x, 'Label'] = key
                        break
                else:
                    # only if there is a label that can't be inferred
                    found_mismatch = True 
    if found_mismatch:
        print(f'Labels need to be corrected to match any of {list(inferable_metadata["Label"].keys())}')
        print(f"Current labels are {complete_df['Label'].unique()}")

# use inferable metadata to fill in missing metadata for each label and cell line
for column in ["Cell_Line_Name", "Label"]:
    for entry in inferable_metadata[column]:
        for inferred_column in inferable_metadata[column][entry]:
            if not inferred_column in complete_df.columns:
                complete_df[inferred_column] = np.nan
                complete_df[inferred_column] = complete_df[inferred_column].astype('str')
            complete_df.loc[complete_df[column]==entry, inferred_column] = inferable_metadata[column][entry][inferred_column]


Inferred label AOPI to be AO_Red based on alternative names
Inferred label AOGFP to be AO_Green based on alternative names
Inferred label LT to be Lysotracker based on alternative names


In [19]:
# add per label excitation and emission values
# can delete if all values are unknown
if set(complete_df['Label'].unique()) == ex_em_fluor_dict.keys():
    for key in ex_em_fluor_dict:
        complete_df.loc[complete_df["Label"] == key, "Microscope_Excitation_Peak"] = ex_em_fluor_dict[key]['ex_peak']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Excitation_Width"] = ex_em_fluor_dict[key]['ex_width']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Emission_Peak"] = ex_em_fluor_dict[key]['em_peak']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Emission_Width"] = ex_em_fluor_dict[key]['em_width']
        complete_df.loc[complete_df["Label"] == key, "Label_Fluorophore"] = ex_em_fluor_dict[key]['fluorophore']
else:
    print("Excitation/Emission dictionary does not match Labels")
    print(f"Available labels are {set(complete_df['Label'].unique())}")
    print(f"Defined keys are {ex_em_fluor_dict.keys()}")

In [44]:
# manual harmonization for this dataset
complete_df.loc[complete_df['NPSize_nm']=="40","Treatment_Primary_Treatment"] = 'AGNP_40nm'
complete_df.loc[complete_df['NPSize_nm']=="100","Treatment_Primary_Treatment"] = 'AGNP_100nm'
complete_df['Timepoint_Primary_Treatment'] = complete_df['Time_hr']
complete_df.loc[complete_df['Treatment_Control_Class']!='NegCon','Treatment_Control_Class'] = np.nan

In [45]:
# report on un-harmonized columns
# use reported information to manually update ontology OR dataframe OR input metadata
# this cell does NOT save the harmonization results, just check them
# this allows you to make any necessary corrections without accidentally overwriting
# returns a view of what the dataframe will look like after final harmonization
extra_cols = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df, ret="extra_cols")

print("View of data that will be kept")
complete_df[[x for x in complete_df.columns if x not in extra_cols]]

Removing columns not in ontology: ['NPSize_nm', 'User_Name', 'Dye', 'DILI-concern', 'Time_hr', 'Plate_Map', 'Severity Class', 'Label', 'Cell']
Adding missing ontology columns: ['Image_Position_Z', 'Treatment_Broad_Sample', 'Year_Imaged', 'Treatment_InChIKey', 'Timepoint_Secondary_Treatment', 'Treatment_Solvent', 'Treatment_SMILES', 'Treatment_Secondary_Treatment', 'Treatment_PubChem_CID']
View of data that will be kept


,Well,Site,Plate,Batch,File Path,File Name,Treatment_Primary_Treatment,Treatment_Concentration,Treatment_Control_Class,Treatment_Mechanism,...,Label_Reagent,Label_Structure,Label_Molecule,Label_Mechanism,Microscope_Excitation_Peak,Microscope_Excitation_Width,Microscope_Emission_Peak,Microscope_Emission_Width,Label_Fluorophore,Timepoint_Primary_Treatment
0,B10,10,211008_092824_Plate_1,2021_10_08_AgNPViability,https://cellpainting-gallery.s3.us-east-1.amaz...,B10_02_2_10_Propidium Iodide_001,AGNP_40nm,0.0013,<NA>,<NA>,...,Acridine Orange,"Lysosomes, Endosomes",Acidic Compartments,Dye,531,<NA>,647,<NA>,Acridine Orange,<NA>
1,B10,11,211008_092824_Plate_1,2021_10_08_AgNPViability,https://cellpainting-gallery.s3.us-east-1.amaz...,B10_02_2_11_Propidium Iodide_001,AGNP_40nm,0.0013,<NA>,<NA>,...,Acridine Orange,"Lysosomes, Endosomes",Acidic Compartments,Dye,531,<NA>,647,<NA>,Acridine Orange,<NA>
2,B10,12,211008_092824_Plate_1,2021_10_08_AgNPViability,https://cellpainting-gallery.s3.us-east-1.amaz...,B10_02_2_12_Propidium Iodide_001,AGNP_40nm,0.0013,<NA>,<NA>,...,Acridine Orange,"Lysosomes, Endosomes",Acidic Compartments,Dye,531,<NA>,647,<NA>,Acridine Orange,<NA>
3,B10,13,211008_092824_Plate_1,2021_10_08_AgNPViability,https://cellpainting-gallery.s3.us-east-1.amaz...,B10_02_2_13_Propidium Iodide_001,AGNP_40nm,0.0013,<NA>,<NA>,...,Acridine Orange,"Lysosomes, Endosomes",Acidic Compartments,Dye,531,<NA>,647,<NA>,Acridine Orange,<NA>
4,B10,14,211008_092824_Plate_1,2021_10_08_AgNPViability,https://cellpainting-gallery.s3.us-east-1.amaz...,B10_02_2_14_Propidium Iodide_001,AGNP_40nm,0.0013,<NA>,<NA>,...,Acridine Orange,"Lysosomes, Endosomes",Acidic Compartments,Dye,531,<NA>,647,<NA>,Acridine Orange,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16797,G7,5,220408_131035_Plate_1,2022_03_24_Acidification_LT,https://cellpainting-gallery.s3.us-east-1.amaz...,G7_03_1_5_DAPI_001,Chloroquine,<NA>,<NA>,autophagy inhibitor,...,Hoechst,Nucleus,DNA,Dye,377,<NA>,447,<NA>,33342,4.0
16798,G7,6,220408_131035_Plate_1,2022_03_24_Acidification_LT,https://cellpainting-gallery.s3.us-east-1.amaz...,G7_03_1_6_DAPI_001,Chloroquine,<NA>,<NA>,autophagy inhibitor,...,Hoechst,Nucleus,DNA,Dye,377,<NA>,447,<NA>,33342,4.0
16799,G7,7,220408_131035_Plate_1,2022_03_24_Acidification_LT,https://cellpainting-gallery.s3.us-east-1.amaz...,G7_03_1_7_DAPI_001,Chloroquine,<NA>,<NA>,autophagy inhibitor,...,Hoechst,Nucleus,DNA,Dye,377,<NA>,447,<NA>,33342,4.0
16800,G7,8,220408_131035_Plate_1,2022_03_24_Acidification_LT,https://cellpainting-gallery.s3.us-east-1.amaz...,G7_03_1_8_DAPI_001,Chloroquine,<NA>,<NA>,autophagy inhibitor,...,Hoechst,Nucleus,DNA,Dye,377,<NA>,447,<NA>,33342,4.0


In [46]:
print("View of data that will be removed")
complete_df[extra_cols]

View of data that will be removed


,NPSize_nm,User_Name,Dye,DILI-concern,Time_hr,Plate_Map,Severity Class,Label,Cell
0,40,URL_AOPI,NaN,NaN,NaN,platemap_agnp1,NaN,AO_Red,Huh7
1,40,URL_AOPI,NaN,NaN,NaN,platemap_agnp1,NaN,AO_Red,Huh7
2,40,URL_AOPI,NaN,NaN,NaN,platemap_agnp1,NaN,AO_Red,Huh7
3,40,URL_AOPI,NaN,NaN,NaN,platemap_agnp1,NaN,AO_Red,Huh7
4,40,URL_AOPI,NaN,NaN,NaN,platemap_agnp1,NaN,AO_Red,Huh7
...,...,...,...,...,...,...,...,...,...
16797,NaN,URL_OrigDNA,LT,NaN,4.0,platemap_131035,NaN,DNA,Huh7
16798,NaN,URL_OrigDNA,LT,NaN,4.0,platemap_131035,NaN,DNA,Huh7
16799,NaN,URL_OrigDNA,LT,NaN,4.0,platemap_131035,NaN,DNA,Huh7
16800,NaN,URL_OrigDNA,LT,NaN,4.0,platemap_131035,NaN,DNA,Huh7


In [47]:
# After correcting any warnings above, run to save harmonization
complete_df = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df)

Removing columns not in ontology: ['NPSize_nm', 'User_Name', 'Dye', 'DILI-concern', 'Time_hr', 'Plate_Map', 'Severity Class', 'Label', 'Cell']
Adding missing ontology columns: ['Image_Position_Z', 'Treatment_Broad_Sample', 'Year_Imaged', 'Treatment_InChIKey', 'Timepoint_Secondary_Treatment', 'Treatment_Solvent', 'Treatment_SMILES', 'Treatment_Secondary_Treatment', 'Treatment_PubChem_CID']


In [48]:
saved_path = os.path.join(output_directory,f"{project_name}_harmonized_metadata_v0_1.parquet")
complete_df.to_parquet(saved_path)